# Colony counting pipeline

Isolate each well from a multi-well plate `.tif` scan, then run each well crop through Cellpose to count colonies and estimate diameter

## Pre-reqs
- All your scans in a given folder should have the same plate/well layout

## How to use this notebook

1. Upload your scans to a Google Drive folder
2. Share the folder as "Anyone with the link"; copy the link
3. Paste it into `FOLDER_URL` in the Config cell below
4. Set `PLATE_ROWS` and `PLATE_COLS` to your plate's well layout
5. **Runtime > Run all.** The first time, the setup cell installs packages and restarts the runtime — Colab reports this as a crash, which is expected. When it reconnects, run all again
6. (Optional) select a different scan to tune with
7. The run will stop at "Convert boxes to ROI hints", this is expected
8. Draw one box per plate on the widget in Section 2
9. Select "Convert boxes to ROI hints" and Runtime > **Run cell and below**

### Changing anything afterwards

**Click the cell you changed, then Runtime > Run cell and below.** 

### The two modes

1. **Tune** (default): downloads and runs the one scan you picked in the Scan dropdown, with
   plots shown, so you can check the counts before committing to a batch
2. **Batch**: downloads every scan in the folder and scans them. It writes results + per-well PNGs to an output .zip file


## 1. Config

**1a: fill in every run** — dataset location and plate layout.
**1b: tuning defaults** — Cellpose segmentation settings are the ones to adjust if your data isn't being counted correctly

In [ ]:
#@title Dataset & plate layout (fill in every run)
# Settled here rather than in the Colab setup cell below, so this cell can validate the
# Colab-only fields without needing that cell (which installs packages and may restart the
# runtime) to have run first. The setup cell reuses it.
import sys

IN_COLAB = "google.colab" in sys.modules

#@markdown ### Run mode
#@markdown **Tune** runs `REFERENCE_SCAN` alone with plots shown, so you can check
#@markdown settings before committing to a batch. **Batch** sweeps every scan in
#@markdown `INPUT_DIR` (Section 2c) and writes results/PNGs to disk.
RUN_MODE = "Tune (single image)"  #@param ["Tune (single image)", "Batch (all scans)"]
BATCH_MODE = RUN_MODE.startswith("Batch")

#@markdown ---
#@markdown ### Dataset location
#@markdown Link to the Drive folder holding your scans, shared "Anyone with the link".
#@markdown (Colab only — ignored when running locally with uv.)
FOLDER_URL = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Plate layout
PLATE_ROWS = None  #@param {type:"raw"}
PLATE_COLS = None  #@param {type:"raw"}

# Fail here, not downstream when these are first used (Section 2c for PLATE_ROWS/
# PLATE_COLS, the Dataset cell for FOLDER_URL) -- catching a blank field a few cells
# later after other setup has already run just wastes a run.
if not PLATE_ROWS or not PLATE_COLS:
    raise ValueError("PLATE_ROWS/PLATE_COLS are unset -- fill them in above and re-run this cell.")
if IN_COLAB and not FOLDER_URL:
    raise ValueError("FOLDER_URL is empty -- paste your shared folder's link above and re-run this cell.")


In [ ]:
#@title Cellpose parameters
# math, not numpy: this cell runs before the Colab install/import cells below, so only
# the stdlib is guaranteed to be importable here.
import math

BATCH_OUTPUT_DIR = "batch_output"  #@param {type:"string"}
# Placement-shift tolerance: how far a plate may sit from where it was drawn, as a
# fraction of the drawn box. This is a property of hand-placing trays in a scanner, not of
# any one dataset, so it is set generously; resolve_axis is what stops a wide search window
# from accepting a wrong edge.
MARGIN_FRAC = 0.20  #@param {type:"slider", min:0.0, max:0.5, step:0.01}

# ============================================================
# Well-detection constants (scan contrast, Hough ring-snap, rigid-geometry tolerances).
# Not exposed as form fields — these tune plate/well *finding*, not colony segmentation,
# and shouldn't need touching unless detection itself misbehaves on a new scan style.
# Edit the values directly (Colab: View > Editor view, or expand this cell) if it does.
# ============================================================
WELLS_ONLY = False  # True: stop after well-detection, skip the slower Cellpose step.
CLAHE_CLIP = 3.0

# Per-well edge refinement (local Hough): the analytic layout is close but not
# pixel-perfect; snap each circle to the real ring via Hough in a small ROI around the
# computed center, constrained near the known radius. Falls back to the analytic circle
# if no good ring is found. Downscaling the ROI before searching (then scaling the found
# circle back up) gives ~12x speedup for a few px of localization noise.
REFINE_WELLS = True
REFINE_SEARCH_FRAC = 0.5
REFINE_RADIUS_TOL = 0.2
REFINE_MAX_SHIFT = 0.5
REFINE_DOWNSCALE_PX = 300
HOUGH_PARAM1 = 100
HOUGH_PARAM2 = 30
# Seed radius for that ring search, as a fraction of the grid pitch (the spacing between
# neighbouring wells). Wells nearly fill their grid cell whatever the format, so one number
# covers a moulded 3x2 tray and a column of loose petri dishes — no per-layout radius. It only
# seeds the search: the radius actually used comes from the rings that lock.
WELL_RADIUS_PITCH_FRAC = 0.45

# Rigid-geometry tolerances. A plate can move between scans but cannot change size, so a
# detected box may differ from the drawn one in position but not in dimensions.
# PLATE_SIZE_TOL: how much size disagreement between two snapped plate edges is still
# blamed on edge-localisation noise rather than on one edge having locked onto the wrong
# thing (a neighbouring tray, an interior rib).
# GRID_PITCH_TOL is the same idea one level down — how far the well spacing fitted from the
# wells that locked may stray from the spacing implied by the plate box before the fit is
# judged bogus and only the grid's origin is taken from the locks.
# MAX_GRID_ROTATION_DEGREES: how skew a tray may lie in the scanner before a fitted well
# grid is judged bogus — trays do sit a degree or two off square, they do not sit sideways.
# LOCK_TRUST_FRAC: how far a well's own ring lock may sit from the fitted grid and still be
# believed, as a fraction of the well pitch. Beyond it the lock is something else in the well
# — a dense colony mass reads as a circle — and the grid position is used instead.
PLATE_SIZE_TOL = 0.15
GRID_PITCH_TOL = 0.25
MAX_GRID_ROTATION_DEGREES = 8.0
LOCK_TRUST_FRAC = 0.25

# Well numbering order, handled by wellcrop's label_scheme. "row-major" counts across
# each row first (A1 A2 / A3 A4 / A5 A6 on a 3x2 plate); "column-major" counts down
# each column first — the lab convention here: left column A1-A3, right column A4-A6.
WELL_LABEL_SCHEME = "column-major"

# ============================================================
# Cellpose colony segmentation. Tune these if colonies look under/over-segmented (two
# touching colonies merged into one blob, or one colony split into two). Full parameter
# docs: https://cellpose.readthedocs.io/en/latest/settings.html
# (API reference for the eval() call these feed: https://cellpose.readthedocs.io/en/latest/api.html)
# ============================================================
#@markdown ---
#@markdown ### Cellpose colony segmentation
#@markdown Parameter docs: [cellpose.readthedocs.io/settings](https://cellpose.readthedocs.io/en/latest/settings.html)
# Expected colony diameter in px, in the LAB-distance "signal" image fed to Cellpose (not
# the raw well crop). Cellpose uses this to scale its internal model — too large merges
# nearby colonies, too small can shatter one colony into several. A well crop on these scans
# is ~1900 px across and a distinct colony ~80-120 px, so this is about one colony; raising
# it is the knob that makes a cloud of specks read as a single object rather than a crowd.
#@markdown **Colony diameter (px):** too large merges touching colonies; too small splits one colony into several.
CELLPOSE_DIAMETER = 70  #@param {type:"integer"}
# Max allowed flow-reconstruction error per mask (Cellpose's internal QC score). Higher =
# keep more masks, including rougher/noisier ones; lower = reject malformed masks more
# aggressively (fewer false positives, but can also drop real irregular colonies).
#@markdown **Flow threshold:** higher keeps more (rougher) masks; lower rejects malformed ones more aggressively.
CELLPOSE_FLOW_THRESHOLD = 0.9  #@param {type:"slider", min:0.0, max:3.0, step:0.01}
# A pixel is called "part of a colony" where the model's cell-probability map exceeds this.
# Lower (more negative) = more permissive, catches faint/sparse colonies but risks noise;
# higher = stricter, cleaner background but can miss faint real colonies.
#@markdown **Cell-probability threshold:** lower (more negative) catches faint colonies but risks noise; higher is stricter.
CELLPOSE_CELLPROB_THRESHOLD = -2.0  #@param {type:"slider", min:-6.0, max:6.0, step:0.1}
# Percentile range used to normalize the signal image's intensity before segmentation —
# clips extreme outlier pixels so one bright artifact doesn't wash out the contrast Cellpose
# needs to see real colonies.
#@markdown **Normalize percentile range:** clips extreme-outlier pixels before segmentation.
CELLPOSE_NORM_LOW = 1.0  #@param {type:"slider", min:0.0, max:10.0, step:0.5}
CELLPOSE_NORM_HIGH = 99.0  #@param {type:"slider", min:90.0, max:100.0, step:0.5}
CELLPOSE_NORMALIZE_PERCENTILE = [CELLPOSE_NORM_LOW, CELLPOSE_NORM_HIGH]
# Smallest thing still counted as a colony, given as its diameter in px of the well crop and
# converted to the mask *area* Cellpose wants (min_size). This is the pin-prick filter: a
# single-cell fleck reads ~10 px across, far under any real colony, and no probability
# threshold rejects it as reliably as an outright size cut — a speck can be perfectly
# confident and still not be a colony. Because Cellpose runs its dynamics at the original crop
# resolution (resample=True), this is in raw crop pixels, not rescaled ones. Raise it if specks
# still get counted; lower it if genuinely small but real colonies start disappearing.
#@markdown **Minimum colony diameter (px):** filters out single-cell specks; raise if debris still counts, lower if small real colonies vanish.
CELLPOSE_MIN_COLONY_DIAMETER = 15  #@param {type:"integer"}
CELLPOSE_MIN_SIZE = int(math.pi * (CELLPOSE_MIN_COLONY_DIAMETER / 2) ** 2)
# Label each segmented colony with its mask number on the AI panel. Off by default --
# numbers are for debugging merges/splits, not for the results view.
SHOW_COLONY_LABELS = False  #@param {type:"boolean"}
# Non-colony brightness outlier rejection, in LAB L (lightness) units, relative to the local
# well-background L — not tied to position, since glints/debris can land anywhere in a well.
# Crystal-violet colonies sit in a mid-darkness band: dirt/hair/specks are much darker
# (near-black) than any real colony, while a specular reflection/glint is brighter than
# background. (dark_margin, bright_margin): a pixel darker than background by more than
# dark_margin, or brighter than background by more than bright_margin, is zeroed out of the
# Cellpose signal image before segmentation. Raise dark_margin if real dense/dark colonies
# start getting stripped; raise bright_margin if real bright-background wells start losing
# edge pixels; lower either if debris/glint still leaks through as false colonies.
#@markdown ---
#@markdown ### Debris/glint filtering (pre-Cellpose, not a Cellpose parameter)
#@markdown Rejects outlier pixels from the signal image *before* Cellpose ever sees it:
#@markdown dark = debris cutoff, bright = glint cutoff, relative to well background.
#@markdown Higher = more permissive (only the most extreme pixels on that side get
#@markdown rejected); lower = stricter (rejects more, risking real colony pixels too).
L_DARK_MARGIN = 97  #@param {type:"slider", min:0, max:150, step:1}
L_BRIGHT_MARGIN = 40  #@param {type:"slider", min:0, max:150, step:1}
L_OUTLIER_MARGIN = (L_DARK_MARGIN, L_BRIGHT_MARGIN)  # (dark_margin, bright_margin)

# ============================================================
# Background reference (scanner-pinned). The signal image measures every pixel's
# distance from "the well's background color"; that reference is picked per well as
# the histogram MODE of the well's interior core. These constants pin what a valid
# background looks like on this lab's scanner (same device for all scans): bare
# plastic is near-neutral light blue-grey. Measured across dense and sparse wells on
# real scans: mode chroma lands within ~5 LAB units of (128.5, 114.0) and core median
# L within 166-188, while a stain-dominated core (the mode landing on colony purple,
# e.g. ~(140, 99)) sits ~18+ units away. If a well's picked background strays outside
# these bounds, its signal zero-point can't be trusted -- the well is failed loudly
# (count 0 + error in the output image and CSV) instead of emitting garbage counts.
BG_CHROMA_REF = (128.5, 114.0)  # (a, b): this scanner's bare-plastic chroma
BG_CHROMA_TOL = 10.0            # LAB units; mode farther than this from REF fails the well
BG_L_MIN = 145.0                # core median lightness below this is not "whiteish" -> fail
# Interior-core sampling: how much of the well radius to erode away before sampling
# the background chroma. The rim ring this replaced is where stain pools (meniscus),
# so the core must stay clear of the edge. 0.31 of the radius on these scans (~240px
# of a ~780px crop) clears the stained meniscus with margin; the floor keeps small
# wells from eroding away entirely.
BG_CORE_ERODE_FRAC = 0.31
BG_CORE_ERODE_MIN_PX = 31
# Histogram window for the mode search, in LAB a/b units. Pinned to the scanner: wide
# enough for every plastic/tint seen so far (a 125-145, b 94-119), tight enough that a
# runaway mode cannot hide far from REF without tripping BG_CHROMA_TOL.
BG_MODE_A_RANGE = (110, 150)
BG_MODE_B_RANGE = (70, 120)
# Crystal-violet test (scanner-pinned): a real colony is redder AND bluer than the
# well background (a up, b down), while wall/trough reflections are chromatically
# neutral yet blobby -- in sparse wells, where no strong colony texture competes,
# Cellpose anchors phantom masks on them (B5/B4 empty-well rim junk). Each mask is
# tested on its most-violet pixels -- the 90th-percentile a-shift -- so dense-lawn
# masks whose mean chroma is diluted by washed background are not hit (dense wells
# lose ~0-1% of masks; sparse-well phantoms lose ~all). Measured gap on this
# scanner: real colonies p90 a-shift >= 5 (faint B-plate dots ~7), trough phantoms
# <= 3.5.
COLONY_MIN_VIOLET_A = 4.0   # p90 of (mask pixel a - well background a), LAB units
COLONY_MIN_VIOLET_B = 0.0   # median of (well background b - mask pixel b), LAB units
 
print("Config loaded. Plate geometry now comes from ROI hints (Section 2c), not a profile.")


## Colab, Device & Model setup


In [ ]:
#@title Setup (Colab clone + install)
# IN_COLAB comes from the Config cell (1a) above.
if IN_COLAB:
    import importlib
    import importlib.metadata
    import os
    import sys
    import time

    REPO_URL = "https://github.com/Districtfine/auto-clonogenics.git"
    REPO_DIR = "/content/auto-clonogenics"  # absolute -- re-running this cell from inside
    # the repo (cwd already moved) must not treat "auto-clonogenics" as a fresh relative
    # target and clone into itself again, which nests a new copy each re-run.

    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    SETUP_MARKER = "/content/.setup_complete"

    if os.path.exists(SETUP_MARKER):
        print("Packages already installed earlier this session -- carrying on.")
    else:
        # Colab already ships torch/opencv/numpy/pandas/scipy with GPU wiring intact --
        # only pull the packages this pipeline adds on top, so we don't clobber those.
        # numba (pulled in transitively by cellpose) caps numpy below its own ceiling, and
        # letting pip resolve past that leaves numba installed against a numpy it declares
        # incompatible with (pip warns but doesn't block), which surfaces later as an
        # ImportError deep in an unrelated cell. Pin numpy up front so the resolver can't
        # drift. The ceiling must match whatever numba version Colab's base image actually
        # preinstalls (check the pip resolver's own "numba X.Y.Z requires numpy<A.B"
        # warning if this breaks again) -- too tight a ceiling has no wheel yet for a new
        # Colab Python bump and silently falls back to a slow from-source numpy build.
        import numpy
        numpy_in_memory = numpy.__version__

        !pip install -q "numpy<2.3" cellpose tifffile jupyter-bbox-widget gdown "wellcrop>=0.2.0"

        # Read the version off disk rather than from the imported module: pip may have
        # replaced numpy underneath us, and the module object won't know. invalidate_caches
        # drops the directory listings cached from before pip wrote the new dist-info.
        importlib.invalidate_caches()
        numpy_on_disk = importlib.metadata.version("numpy")

        if numpy_on_disk == numpy_in_memory:
            # Colab's numpy already satisfied the pin, so nothing was swapped: the module in
            # memory still matches what's on disk, and scipy is still built against it.
            open(SETUP_MARKER, "w").close()
            print(f"Installed. numpy {numpy_on_disk} already satisfied the pin -- "
                  "no restart needed, carry on down the notebook.")
        else:
            # numpy moved. scipy was built against the old one, so rebuild it to match --
            # a mixed-version set of numpy/scipy files on disk is a partial upgrade that
            # fails later in an unrelated cell.
            !pip install -q --force-reinstall "numpy<2.3" scipy
            open(SETUP_MARKER, "w").close()

            # The kernel imported numpy before any of our cells ran, and Python can't unload
            # a C extension -- anything importing against the other ABI segfaults. Kill the
            # process so Colab hands us a fresh one that loads what's now on disk.
            print("=" * 70)
            print(f"numpy {numpy_in_memory} -> {numpy_on_disk}. Restarting the runtime once "
                  "so numpy/scipy load cleanly.")
            print("Colab will say the session crashed -- that's this restart, not a failure.")
            print("When it reconnects: Runtime > Run all again. Nothing reinstalls this time.")
            print("=" * 70)
            # SIGKILL drops anything ipykernel hasn't pushed to the browser yet, which would
            # take the message above with it and leave only Colab's "crashed for an unknown
            # reason". Flush, then give the stream a moment to actually go out.
            sys.stdout.flush()
            time.sleep(2)
            os.kill(os.getpid(), 9)
else:
    print("Not running in Colab — skipping clone/install (using local uv environment).")


In [ ]:
#@title Imports
import os
import glob
import shutil
import warnings
from datetime import datetime
warnings.filterwarnings("ignore", message="Sparse invariant checks")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tifi
import torch
from scipy import ndimage
from tqdm.notebook import tqdm
from cellpose import models
from cellpose import transforms as cp_transforms
from wellcrop import PlateDetector, render_overlay_matplotlib, Well

plt.rcParams['figure.figsize'] = [12, 6]


In [ ]:
#@title Pick device & load model
# Portable device pick: mps on Apple Silicon, cuda on the 1650 Ti box, cpu fallback.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
TORCH_DEVICE = torch.device(DEVICE)
print(f"Using device: {DEVICE}")

if "cp_model" in globals():
    # Reloading this is slow (weights + GPU transfer) and pointless once loaded --
    # "Run this cell and below" after tweaking a Cellpose param shouldn't reload the
    # model too. Restart the runtime if you actually need a clean reload.
    print("Model already loaded, reusing.")
else:
    print("Loading Cellpose (Colony Counter) into GPU...")
    cp_model = models.CellposeModel(gpu=DEVICE != "cpu", device=TORCH_DEVICE, model_type='cpsam_v2')
    # Directly confirm what device the model actually landed on — don't infer this from
    # whether Cellpose's own log messages showed up, since those need io.logger_setup()
    # (which also drives a per-tile progress bar) to be visible at all.
    print(f"cp_model.device = {cp_model.device}, cp_model.gpu = {cp_model.gpu}")
    print("Done!")


## 2. Draw ROI hints

- **2a** lists your Drive folder (no downloads) and offers the Scan dropdown.
- **2b** fetches what the run mode needs: in Tune just the scan you picked, in Batch the whole folder.
- **2c** turns your drawn boxes into ROI hints, reused across every scan in a batch.

Draw one box per plate on the widget — any order, they're labelled A, B, … top to bottom.
The drawing is kept across re-runs and even when you tune against a different scan
(pick it in the dropdown, Runtime > **Run cell and below**) — boxes are fractions of the
image, so they carry over; nudge them only if the new scan's plates sit elsewhere.


In [ ]:
#@title Select scan (for tuning)
# ============================================================
# Dataset — which scans to run. ROI hints are drawn once on REFERENCE_SCAN (below) and
# reused for every scan in INPUT_DIR, so one folder must hold one plate layout; change
# FOLDER_URL in the Config cell (Section 1) and redraw to switch datasets.
# ============================================================
if not IN_COLAB:
    INPUT_DIR = "../Clonogenics"           # <-- edit to your local scans folder
    REFERENCE_SCAN = "reference_scan.tif"  # <-- edit to your reference scan filename
    BATCH_SCANS = []

else:
    # Reads a Drive folder shared as "Anyone with the link" -- no OAuth, no whole-account
    # grant, only the one folder pointed to by FOLDER_URL.
    import gdown
    import ipywidgets as widgets
    from IPython.display import display

    shared_download_dir = "/content/shared_scans"
    os.makedirs(shared_download_dir, exist_ok=True)

    same_folder = globals().get("_listed_folder_url") == FOLDER_URL
    if same_folder and "remote_scans" in globals():
        # Same folder as last time -- reuse the listing instead of re-walking Drive just
        # because a Cellpose param changed and this cell re-ran.
        print(f"Already listed this folder ({len(remote_scans)} scans) -- reusing.")
    else:
        # List only, download nothing: these are ~100MB TIFFs, and a tune run needs exactly
        # one of them. The next cell fetches what the run mode actually calls for. Only
        # .tif/.tiff are scans -- folders synced from a Mac often carry .DS_Store etc.
        remote_files = gdown.download_folder(FOLDER_URL, skip_download=True)
        remote_scans = {
            os.path.basename(item.path): item.id
            for item in remote_files
            if item.path.lower().endswith((".tif", ".tiff"))
        }
        _listed_folder_url = FOLDER_URL
        print(f"Found {len(remote_scans)} scans in the shared folder (nothing downloaded yet).")

    if not remote_scans:
        raise ValueError(
            "No .tif/.tiff files found at FOLDER_URL -- check the link and that the folder "
            "is shared \"Anyone with the link\"."
        )

    # Re-run of this cell (Run all, or "Run cell and below"): Colab disposes the old
    # widget when its output is cleared, so redisplaying the same instance renders
    # nothing. Rebuild the dropdown instead, but carry over the picked scan so a
    # re-run doesn't reset the selection to the first option.
    previous_pick = scan_picker.value if same_folder and "scan_picker" in globals() else None
    picker_options = sorted(remote_scans)
    picker_kwargs = {"description": "Scan:", "options": picker_options}
    if previous_pick in picker_options:
        picker_kwargs["value"] = previous_pick
    scan_picker = widgets.Dropdown(**picker_kwargs)
    display(scan_picker)


In [ ]:
#@title Fetch Scans
import re
if IN_COLAB:
    # Lock in what got picked in the dropdown above. Downloads land in a per-folder
    # subdirectory of the shared workspace: keyed by Drive folder, so a re-run against
    # a different FOLDER_URL can't collide with (or batch-list) the previous folder's
    # files. Flat storage used to leak folder A's leftovers into folder B's batch --
    # "already downloaded" matched by filename, and the sweep listed the whole dir.
    folder_label = remote_files[0].path.split("/")[0] if remote_files else "drive_folder"
    folder_tag = re.sub(r"[^A-Za-z0-9._-]+", "_", folder_label).strip("_") or "drive_folder"
    INPUT_DIR = os.path.join(shared_download_dir, folder_tag)
    os.makedirs(INPUT_DIR, exist_ok=True)
    REFERENCE_SCAN = scan_picker.value
    BATCH_SCANS = []  # [] runs every .tif/.tiff in INPUT_DIR; or list filenames for a subset

    # Tune needs one scan, batch needs the folder -- fetch accordingly instead of pulling
    # every ~100MB TIFF just to look at one. Downloading file-by-file also means one file
    # tripping Drive's per-file rate limit ("had many accesses" -- easy to hit while
    # iterating on the same shared link) gets skipped instead of aborting the whole batch.
    wanted_scans = sorted(remote_scans) if BATCH_MODE else [REFERENCE_SCAN]
    for scan_name in wanted_scans:
        local_path = os.path.join(INPUT_DIR, scan_name)
        if os.path.exists(local_path):
            print(f"{scan_name} already downloaded -- reusing.")
            continue
        try:
            gdown.download(id=remote_scans[scan_name], output=local_path, quiet=False)
        except Exception as error:
            print(f"!! Skipping {scan_name}: {error}")

    # A tune run reads REFERENCE_SCAN directly in the next cell, so a failed download has to
    # surface here rather than as a file-not-found three cells later.
    if not os.path.exists(os.path.join(INPUT_DIR, REFERENCE_SCAN)):
        raise RuntimeError(
            f"{REFERENCE_SCAN} failed to download -- see the error above, then re-run this cell."
        )

print(f"INPUT_DIR = {INPUT_DIR!r}")
print(f"REFERENCE_SCAN = {REFERENCE_SCAN!r}")


In [ ]:
#@title Draw ROI boxes
from jupyter_bbox_widget import BBoxWidget
from IPython.display import display

# Grab the current drawing (if any) before rebuilding: Colab disposes the old widget
# when this cell's output is cleared, so redisplaying the same instance renders
# nothing. We build a live widget on every run and carry the boxes over instead.
prev_bboxes = bbox_widget.bboxes if "bbox_widget" in globals() else []
prev_shape = globals().get("_preview_shape")

if globals().get("_bbox_reference_scan") == REFERENCE_SCAN:
    # Same reference scan as last time -- reuse the cached preview file (and the
    # in-memory copy the Convert cell below divides by) instead of re-reading the
    # ~100MB TIFF just because this cell re-ran.
    preview_path = "roi_preview.png"
    print(f"Reusing cached preview ({len(prev_bboxes)} boxes carried over) -- "
          "tweak the drawing and re-run Convert below if needed.")
else:
    # Load the reference scan, downscale to a displayable 8-bit preview for the widget.
    first_scan = tifi.imread(os.path.join(INPUT_DIR, REFERENCE_SCAN))
    first_rgb = cv2.cvtColor(first_scan, cv2.COLOR_GRAY2RGB) if first_scan.ndim == 2 else first_scan[..., :3]
    if first_rgb.dtype != np.uint8:
        first_rgb = cv2.normalize(first_rgb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    preview_scale = 1200 / max(first_rgb.shape[0], first_rgb.shape[1])
    preview = cv2.resize(first_rgb, None, fx=preview_scale, fy=preview_scale, interpolation=cv2.INTER_AREA)
    preview_path = "roi_preview.png"
    cv2.imwrite(preview_path, cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))
    _bbox_reference_scan = REFERENCE_SCAN
    _preview_shape = preview.shape[:2]

# Boxes are stored in preview pixels; if the new scan's preview is a different size,
# rescale the drawing. ROI hints are fractions of the image, so one drawing transfers
# across every scan in the folder.
if prev_bboxes and prev_shape != preview.shape[:2]:
    scale_y = preview.shape[0] / prev_shape[0]
    scale_x = preview.shape[1] / prev_shape[1]
    prev_bboxes = [
        {**box, "x": box["x"] * scale_x, "y": box["y"] * scale_y,
         "width": box["width"] * scale_x, "height": box["height"] * scale_y}
        for box in prev_bboxes
    ]

# hide_buttons=True drops the Submit/Skip buttons (we read bbox_widget.bboxes directly,
# no submit callback needed). Always a fresh widget so it renders and stays tweakable.
bbox_widget = BBoxWidget(image=preview_path, classes=["plate"], hide_buttons=True, bboxes=prev_bboxes)
display(bbox_widget)


In [ ]:
#@title Convert boxes to ROI hints
# Convert preview-pixel boxes to image fractions, top-to-bottom, labeled A, B, ...
drawn_boxes = sorted(bbox_widget.bboxes, key=lambda box: box["y"])
roi_hints = []
for plate_index, box in enumerate(drawn_boxes):
    roi_hints.append({
        "x": box["x"] / preview.shape[1],
        "y": box["y"] / preview.shape[0],
        "w": box["width"] / preview.shape[1],
        "h": box["height"] / preview.shape[0],
        "rows": PLATE_ROWS,
        "cols": PLATE_COLS,
        "letter": chr(ord("A") + plate_index),
    })
if not roi_hints:
    raise ValueError(
        "No boxes drawn -- draw one box per plate on the bbox widget above, "
        "then re-run this cell."
    )

print(f"Captured {len(roi_hints)} ROI hints:")
for roi in roi_hints:
    print(roi)


## 3. Segment colonies per well (Cellpose)


In [ ]:
#@title Colony segmentation
def count_colonies(wells, plate_name, show_plots=True, save_dir=None):
    """Segment colonies in each detected well and return per-well colony counts.

    show_plots displays each well's raw/segmentation panel inline; save_dir (if given)
    writes one PNG per well to disk instead/as well.
    Wells whose background reference fails the scanner-pinned whiteness check are
    skipped (Colonies 0) and reported with an "Error" message instead of a count.
    """
    report_data = []
    print(f"Extracting wells and generating colony masks for {plate_name}...\n")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    dark_margin, bright_margin = L_OUTLIER_MARGIN

    well_progress = tqdm(wells, desc=plate_name, unit="well")
    for well in well_progress:
        well_progress.set_postfix(well=well.label)
        final_well = well.image   # circular-masked RGB crop from wellcrop
        mask = well.mask

        # --- Color-agnostic signal: distance from background in LAB ---
        lab = cv2.cvtColor(final_well, cv2.COLOR_RGB2LAB).astype(np.float32)

        # Background reference = the most common chroma in the well's interior core.
        # This used to be the median of a 60px rim ring, on the theory that the ring is
        # the one band guaranteed free of colonies. On stained scans that backfires:
        # crystal violet pools at the meniscus, so the ring's median lands ON the
        # colony chroma (measured ~5.7 LAB units from colonies vs ~9.9 from the true
        # background), inverting the signal and shattering dense-well counts (A5
        # counted 10 of ~400). The interior core stays clear of the pooled stain, and
        # the MODE (not the median) survives dense wells: "purple" smears across many
        # chroma values while the pale wash between colonies is the single tight
        # cluster (A6 core: ~51% of pixels within 6 units of the wash vs ~36% within
        # 6 units of colony chroma).
        erode_px = max(BG_CORE_ERODE_MIN_PX, int(well.radius * BG_CORE_ERODE_FRAC))
        core = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                         (erode_px, erode_px)))
        core_ab = lab[:, :, 1:3][core > 0]
        hist2d, a_edges, b_edges = np.histogram2d(
            core_ab[:, 0], core_ab[:, 1],
            bins=[np.arange(BG_MODE_A_RANGE[0], BG_MODE_A_RANGE[1] + 1),
                  np.arange(BG_MODE_B_RANGE[0], BG_MODE_B_RANGE[1] + 1)])
        ia, ib = np.unravel_index(np.argmax(hist2d), hist2d.shape)
        bg_a = (a_edges[ia] + a_edges[ia + 1]) / 2
        bg_b = (b_edges[ib] + b_edges[ib + 1]) / 2
        bg_L = np.median(lab[:, :, 0][core > 0])

        # Scanner-pinned sanity check (see BG_CHROMA_REF in the Config cell): if the
        # picked background is not this scanner's near-neutral light plastic, the
        # signal's zero-point is untrustworthy (stain-dominated core, wrong plate
        # type, botched exposure). Fail the well loudly: count 0, message in the
        # CSV's Error column and in this well's output PNG, instead of garbage counts.
        chroma_off = float(np.hypot(bg_a - BG_CHROMA_REF[0], bg_b - BG_CHROMA_REF[1]))
        if chroma_off > BG_CHROMA_TOL or not np.isfinite(bg_L) or bg_L < BG_L_MIN:
            error_msg = (f"background not whiteish: mode a={bg_a:.1f}, b={bg_b:.1f} "
                         f"({chroma_off:.1f} units from ref {BG_CHROMA_REF}), "
                         f"L={bg_L:.0f} (min {BG_L_MIN:g}) -- well too stained or "
                         f"misdetected, count not trustworthy")
            if show_plots or save_dir:
                fig, ax_err = plt.subplots(figsize=(4.5, 4.5))
                ax_err.imshow(final_well)
                ax_err.set_title(f"{well.label}: 0 colonies -- ERROR\n{error_msg}",
                                 fontsize=9, color="red")
                ax_err.axis('off')
                fig.tight_layout()
                if save_dir:
                    fig.savefig(os.path.join(save_dir, f"{well.label}.png"),
                                dpi=150, bbox_inches='tight')
                if show_plots:
                    plt.show()
                else:
                    plt.close(fig)
            report_data.append({"Plate": plate_name, "Well": well.label,
                                "Colonies": 0, "Error": error_msg})
            continue

        dist = np.sqrt((lab[:, :, 1] - bg_a) ** 2 + (lab[:, :, 2] - bg_b) ** 2)
        signal = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        signal = cv2.bitwise_and(signal, signal, mask=mask)

        # Non-colony brightness outlier rejection (debris too dark, glint too bright)
        dark_mask = (lab[:, :, 0] < (bg_L - dark_margin)) & (mask > 0)
        bright_mask = (lab[:, :, 0] > (bg_L + bright_margin)) & (mask > 0)
        # Plain inpaint, no dilation: the paint is no longer load-bearing for
        # correctness -- the cellprob veto below forbids masks over flagged regions.
        # It only keeps flagged pixels from skewing the percentile normalization and
        # the flow field around them. (Paint variants -- zeroing, dilated inpaint,
        # clipped inpaint -- each fixed one artifact class and created another; the
        # veto replaces that ladder. Verified on the 4T1 rim-glint well: dilate+
        # inpaint gave 31 colonies with 26 rim fakes; the veto gives 4, zero fakes.)
        outlier_px = (dark_mask | bright_mask).astype(np.uint8)
        if outlier_px.any():
            signal = cv2.inpaint(signal, outlier_px, 3, cv2.INPAINT_TELEA)

        # Visual check for L_OUTLIER_MARGIN tuning: yellow = debris (too dark), cyan = glint (too bright).
        outlier_overlay = final_well.copy()
        outlier_overlay[dark_mask] = [255, 255, 0]
        outlier_overlay[bright_mask] = [0, 255, 255]

        # --- Cellpose, with cellprob veto ---
        # Mirrors cp_model.eval()'s internals (normalize -> _run_net -> _compute_masks)
        # with one addition: pixels flagged as debris/glint are forbidden from seeding
        # or anchoring a mask -- cellprob is forced below the threshold there and flows
        # are zeroed, so dynamics can never start or cross into them. A colony cannot
        # be made of debris/glint pixels.
        rescale = 30.0 / CELLPOSE_DIAMETER if CELLPOSE_DIAMETER > 0 else 1.0
        niter = int(200 / rescale)
        norm_signal = cp_transforms.normalize_img(
            signal[np.newaxis, ..., np.newaxis], normalize=True,
            percentile=CELLPOSE_NORMALIZE_PERCENTILE)
        dP, cellprob, _ = cp_model._run_net(
            norm_signal, resample=True, rescale=rescale, augment=False,
            batch_size=8, tile_overlap=0.1, bsize=None)
        veto = cv2.dilate(outlier_px, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))) > 0
        if veto.any():
            cellprob = cellprob.copy()
            cellprob[:, veto] = -1e4
            dP[..., veto] = 0
        masks_cp = cp_model._compute_masks(
            (1, signal.shape[0], signal.shape[1]), dP, cellprob,
            flow_threshold=CELLPOSE_FLOW_THRESHOLD,
            cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
            min_size=CELLPOSE_MIN_SIZE, niter=niter)

        # A colony must be crystal violet: redder AND bluer than the well background
        # (a up, b down). Wall/trough reflections are chromatically neutral but blobby,
        # and in sparse wells -- where no strong colony texture competes for the
        # model's attention -- cpsam anchors phantom masks on them. Each mask is
        # tested on its most-violet pixels (90th percentile of the a-shift) so
        # dense-lawn masks whose means are diluted by washed background stay safe:
        # on this scanner real colonies measure p90 a-shift >= 5, trough phantoms
        # <= 3.5. Failed masks are zeroed and not counted.
        keep = np.zeros(int(masks_cp.max()) + 1, bool)
        keep[0] = True
        for cid in np.unique(masks_cp):
            if cid == 0:
                continue
            pm = masks_cp == cid
            da_px = lab[:, :, 1][pm] - bg_a
            db_px = bg_b - lab[:, :, 2][pm]
            if (np.percentile(da_px, 90) >= COLONY_MIN_VIOLET_A
                    and np.median(db_px) >= COLONY_MIN_VIOLET_B):
                keep[cid] = True
        masks_cp[~keep[masks_cp]] = 0
        colony_count = int((np.unique(masks_cp) > 0).sum())
        report_data.append({
            "Plate": plate_name,
            "Well": well.label,
            "Colonies": colony_count,
            "Error": None
        })

        if show_plots or save_dir:
            fig, axes = plt.subplots(1, 3)
            axes[0].imshow(final_well)
            axes[0].set_title(f"Raw Well: {well.label}", fontsize=10)
            axes[0].axis('off')

            axes[1].imshow(outlier_overlay)
            axes[1].set_title("Outliers (yellow=debris, cyan=glint)", fontsize=10)
            axes[1].axis('off')

            # Mask overlay on the raw well crop: light color fill + a per-colony outline
            # in the matching colormap color. Outlines keep the colony texture visible and
            # make boundaries crisp, so merges/splits are judgeable against the real image.
            colony_ids = np.unique(masks_cp)
            colony_ids = colony_ids[colony_ids != 0]
            mask_norm = plt.Normalize(vmin=1, vmax=max(int(masks_cp.max()), 1))
            # Outlines are composited onto a copy of the raw crop (a separate zeros
            # layer would paint black over everything it doesn't outline).
            overlay = final_well.copy()
            for colony_id in colony_ids:
                contours, _ = cv2.findContours(
                    (masks_cp == colony_id).astype(np.uint8),
                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                r, g, b, _ = plt.cm.nipy_spectral(mask_norm(colony_id))
                cv2.drawContours(overlay, contours, -1,
                                 (int(r * 255), int(g * 255), int(b * 255)), 2)
            axes[2].imshow(final_well)
            axes[2].imshow(masks_cp, cmap='nipy_spectral', alpha=0.3)
            axes[2].imshow(overlay)
            axes[2].set_title(f"AI Count: {colony_count} Colonies", fontsize=10)
            axes[2].axis('off')

            if SHOW_COLONY_LABELS:
                colony_ids = np.unique(masks_cp)
                colony_ids = colony_ids[colony_ids != 0]
                if len(colony_ids) > 0:
                    centroids = ndimage.center_of_mass(masks_cp, masks_cp, colony_ids)
                    for colony_id, (centroid_y, centroid_x) in zip(colony_ids, centroids):
                        axes[2].text(centroid_x, centroid_y, str(int(colony_id)),
                                    color='white', fontsize=6, ha='center', va='center',
                                    bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5, linewidth=0))

            fig.tight_layout()
            if save_dir:
                well_path = os.path.join(save_dir, f"{well.label}.png")
                fig.savefig(well_path, dpi=150, bbox_inches='tight')
            if show_plots:
                plt.show()
            else:
                plt.close(fig)

    return report_data


## 4. Run

Counts colonies in every well and prints the results table. In Batch, the counts and per-well
images download together as a zip when it finishes.


In [ ]:
#@title Run
def process_plate(path, show_plots, save_dir=None, wells_only=False, show_grid=False):
    # show_grid displays the well-selection overlay inline even in batch (where
    # show_plots is off to keep the 12 per-well triptychs from flooding the notebook) --
    # one cheap figure per scan is the fast "are my wells being picked correctly?"
    # feedback users need before committing to a long batch.
    plate_name = os.path.basename(path)
    print(f"Reading {plate_name}...")
    img = tifi.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else img[..., :3]

    # Detect & crop wells with wellcrop
    detector = PlateDetector(
        margin_frac=MARGIN_FRAC,
        clahe_clip=CLAHE_CLIP,
        refine_wells=REFINE_WELLS,
        radius_pitch_frac=WELL_RADIUS_PITCH_FRAC,
        plate_size_tol=PLATE_SIZE_TOL,
        grid_pitch_tol=GRID_PITCH_TOL,
        max_grid_rotation_degrees=MAX_GRID_ROTATION_DEGREES,
        lock_trust_frac=LOCK_TRUST_FRAC,
        hough_param1=HOUGH_PARAM1,
        hough_param2=HOUGH_PARAM2,
        label_scheme=WELL_LABEL_SCHEME,
    )
    wells, plate_boxes = detector.detect(img_rgb, roi_hints, return_boxes=True)

    # Grid overlay: saved as grid.png per plate; shown inline when show_plots (tune)
    # or show_grid (batch) asks for it.
    if show_plots or save_dir or show_grid:
        figure, axis = plt.subplots(figsize=(10, 10))
        render_overlay_matplotlib(
            axis, img_rgb, roi_hints, wells, plate_name,
            plate_boxes=plate_boxes, margin_frac=MARGIN_FRAC
        )
        if save_dir:
            grid_path = os.path.join(save_dir, plate_name, "grid.png")
            os.makedirs(os.path.dirname(grid_path), exist_ok=True)
            figure.savefig(grid_path, dpi=150, bbox_inches="tight")
        if show_plots or show_grid:
            plt.show()
        else:
            plt.close(figure)

    if wells_only:
        return [{"Plate": plate_name, "Well": w.label, "x": w.x, "y": w.y, "r": w.radius}
                for w in wells]

    # Extract circular-masked crop for each well
    for well in wells:
        well.extract_crop(img_rgb)

    wells_dir = os.path.join(save_dir, plate_name) if save_dir else None
    return count_colonies(wells, plate_name, show_plots=show_plots, save_dir=wells_dir)


if BATCH_MODE:
    # Wipe any previous batch_output before this run
    shutil.rmtree(BATCH_OUTPUT_DIR, ignore_errors=True)
    os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

    scan_names = BATCH_SCANS or sorted(
        name for name in os.listdir(INPUT_DIR) if name.lower().endswith((".tif", ".tiff")))
    tif_paths = [os.path.join(INPUT_DIR, scan_name) for scan_name in scan_names]
    n_imgs = len(tif_paths)
    print(f"Batch mode: {n_imgs} plates")
    print(f"Writing well/grid images to {BATCH_OUTPUT_DIR}/<plate_name>/\n")
    report_data = []
    for img_idx, path in enumerate(tif_paths, start=1):
        print(f"\n=== [{img_idx}/{n_imgs}] {os.path.basename(path)} ===")
        report_data.extend(process_plate(
            path, show_plots=WELLS_ONLY, save_dir=BATCH_OUTPUT_DIR,
            wells_only=WELLS_ONLY, show_grid=True
        ))
else:
    report_data = process_plate(os.path.join(INPUT_DIR, REFERENCE_SCAN),
                                show_plots=True, wells_only=WELLS_ONLY)


In [ ]:
#@title Save results
# One parameters.json per run, two audiences:
#   - the top level is the human-readable summary -- mode, commit, every tunable -- the
#     part the lab can read out over the phone when something looks wrong;
#   - "diagnostics" is the forensics layer for "it's not doing the same thing": input
#     checksums, package versions, device, model identity, hardcoded pipeline constants.
import json
import importlib.metadata
import platform
import re
import subprocess

df = pd.DataFrame(report_data)

display(df)

# Name outputs after the dataset folder, not "batch" -- the results belong to it.
# On Colab the folder name sits in the remote listing paths; locally it's INPUT_DIR's.
def _folder_name():
    if IN_COLAB:
        for item in remote_files:
            parent = os.path.dirname(item.path.rstrip("/"))
            if parent:
                return os.path.basename(parent.rstrip("/"))
        return "drive_folder"
    return os.path.basename(os.path.normpath(INPUT_DIR))

folder_name = re.sub(r"[^A-Za-z0-9._-]+", "_", _folder_name()).strip("_") or "scans"
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# "wells" vs "Results" in the name so a wells-only run doesn't overwrite a full run's
# CSV. Batch writes everything inside BATCH_OUTPUT_DIR so it travels with the well/grid
# PNGs in the zip below; tune writes beside the CSV.
suffix = "wells" if WELLS_ONLY else "Results"
if BATCH_MODE:
    csv_name = os.path.join(BATCH_OUTPUT_DIR, f"{folder_name}_{suffix}.csv")
    params_name = os.path.join(BATCH_OUTPUT_DIR, "parameters.json")
else:
    stem = os.path.splitext(os.path.basename(REFERENCE_SCAN))[0]
    csv_name = f"{stem}_{suffix}.csv"
    params_name = f"{stem}_parameters.json"

def _git_commit():
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=10)
        return out.stdout.strip() if out.returncode == 0 else "unknown"
    except Exception:
        return "unknown"

def _pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

def _model_info():
    info = {"file_size_bytes": None}
    try:
        model_path = os.path.expanduser(os.path.join("~", ".cellpose", "models", "cpsam_v2"))
        if os.path.isfile(model_path):
            info["file_size_bytes"] = os.path.getsize(model_path)
        elif os.path.isdir(model_path):
            info["file_size_bytes"] = sum(
                os.path.getsize(os.path.join(model_path, f)) for f in os.listdir(model_path))
    except Exception:
        pass
    return info

parameters = {
    "schema": 1,
    "run_id": run_ts,
    "commit": _git_commit(),
    "mode": RUN_MODE,
    "plate": {
        "rows": PLATE_ROWS,
        "cols": PLATE_COLS,
        "folder_url": FOLDER_URL if IN_COLAB else None,
        "input_dir": None if IN_COLAB else INPUT_DIR,
        "reference_scan": REFERENCE_SCAN,
        "batch_scans": BATCH_SCANS,
    },
    "roi_hints": roi_hints,
    "wellcrop": {
        "margin_frac": MARGIN_FRAC,
        "clahe_clip": CLAHE_CLIP,
        "refine_wells": REFINE_WELLS,
        "refine_search_frac": REFINE_SEARCH_FRAC,
        "refine_radius_tol": REFINE_RADIUS_TOL,
        "refine_max_shift": REFINE_MAX_SHIFT,
        "refine_downscale_px": REFINE_DOWNSCALE_PX,
        "hough_param1": HOUGH_PARAM1,
        "hough_param2": HOUGH_PARAM2,
        "well_radius_pitch_frac": WELL_RADIUS_PITCH_FRAC,
        "plate_size_tol": PLATE_SIZE_TOL,
        "grid_pitch_tol": GRID_PITCH_TOL,
        "max_grid_rotation_degrees": MAX_GRID_ROTATION_DEGREES,
        "lock_trust_frac": LOCK_TRUST_FRAC,
        "label_scheme": WELL_LABEL_SCHEME,
    },
    "cellpose": {
        "diameter": CELLPOSE_DIAMETER,
        "flow_threshold": CELLPOSE_FLOW_THRESHOLD,
        "cellprob_threshold": CELLPOSE_CELLPROB_THRESHOLD,
        "norm_low": CELLPOSE_NORM_LOW,
        "norm_high": CELLPOSE_NORM_HIGH,
        "min_colony_diameter": CELLPOSE_MIN_COLONY_DIAMETER,
    },
    "outliers": {
        "dark_margin": L_DARK_MARGIN,
        "bright_margin": L_BRIGHT_MARGIN,
    },
    "background": {
        "chroma_ref": list(BG_CHROMA_REF),
        "chroma_tol": BG_CHROMA_TOL,
        "L_min": BG_L_MIN,
        "core_erode_frac": BG_CORE_ERODE_FRAC,
        "core_erode_min_px": BG_CORE_ERODE_MIN_PX,
        "mode_a_range": list(BG_MODE_A_RANGE),
        "mode_b_range": list(BG_MODE_B_RANGE),
        "violet_min_a": COLONY_MIN_VIOLET_A,
        "violet_min_b": COLONY_MIN_VIOLET_B,
    },
    "diagnostics": {
        "versions": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "numpy": np.__version__,
            "opencv": cv2.__version__,
            "scipy": __import__("scipy").__version__,
            "cellpose": _pkg_version("cellpose"),
            "wellcrop": _pkg_version("wellcrop"),
        },
        "device": DEVICE,
        "model": {"type": "cpsam_v2", **_model_info()},
        "wells_only": WELLS_ONLY,
        # Constants hardcoded in the pipeline -- not tunables, but they change results
        # when the code changes, so they belong in the forensics record.
        "hardcoded": {
            "outlier_inpaint_radius": 3,
            "cellprob_veto_dilate_kernel_px": [9, 9],
            "cellprob_veto_dilate_shape": "ellipse",
            "cellprob_veto_value": -1e4,
        },
        "outputs": {
            "csv": os.path.basename(csv_name),
            "parameters_json": os.path.basename(params_name),
            "zip": f"{folder_name}_{run_ts}.zip" if BATCH_MODE else None,
        },
    },
}

# Batch writes the JSON inside BATCH_OUTPUT_DIR first so the zip below carries it;
# tune writes it beside the CSV.
with open(params_name, "w") as f:
    json.dump(parameters, f, indent=2)
df.to_csv(csv_name, index=False)

print(f"💾 Data successfully saved to {csv_name}")
print(f"🧾 Run parameters saved to {params_name}")

# Batch is the run whose output people keep, and the VM is ephemeral -- nothing on it
# survives a disconnected/recycled runtime unless it's downloaded, so zip the well/grid
# PNGs, CSV and parameters.json together and push the zip to the browser rather than
# leaving that as a step to forget. Tune is a look-at-the-plots run: no download prompt.
if BATCH_MODE:
    zip_path = shutil.make_archive(f"{folder_name}_{run_ts}", "zip", BATCH_OUTPUT_DIR)
    print(f"📦 Batch output zipped to {zip_path}")

    if IN_COLAB:
        from google.colab import files
        try:
            files.download(zip_path)
        except Exception as error:
            # Chrome refuses the download outright if the tab hasn't been interacted with,
            # or if "allow multiple downloads" was dismissed on an earlier run. Without this
            # the only sign is a traceback, and the results look like they were never produced.
            print(f"!! The download didn't start automatically ({error}).")
            print(f"   Open the Files panel on the left (folder icon), find "
                  f"{os.path.basename(zip_path)}, and download it from there.")
